# Mashtots — учебный разбор решения

Задача: классифицировать изображения армянских рукописных букв, 78 классов
(39 букв × заглавная и строчная), изображения 64×64 в градациях серого,
70 060 штук в обучающей части. Соревнования:
[Mashtots Dataset](https://www.kaggle.com/competitions/mashtots-dataset) и
[Mashtots Dataset v2](https://www.kaggle.com/competitions/mashtots-dataset-v2).

Это учебная версия рабочего ноутбука `mashtots_kaggle.ipynb`. Отличия:

* каждая строка кода снабжена комментарием;
* перед каждым блоком идёт разбор: какие были варианты решения, чем они
  отличаются и почему выбран именно этот;
* добавлены запускаемые демонстрации двух ловушек, на которых в этой задаче
  чаще всего теряют результат (метки классов и порядок строк в submission).

Ноутбук рабочий: его можно запустить целиком и получить `submission.csv`. Но
если нужен просто результат, берите `mashtots_kaggle.ipynb` — он короче.

## Как читать разборы вариантов

В таблицах «вариант / плюсы / минусы» нет универсально правильного ответа —
выбор зависит от размера данных, наличия GPU и того, что важнее: скорость
разработки или последняя доля процента точности. Везде, где выбор неочевиден,
указано условие, при котором стоило бы выбрать иначе.

## 1. Импорты

### Какие были варианты

| Что выбирать | Вариант | Плюсы | Минусы |
|---|---|---|---|
| фреймворк | **Keras/TensorFlow** | аугментация и нормализация встраиваются прямо в модель, поэтому их нельзя забыть на инференсе; меньше кода | меньше контроля над циклом обучения |
| | PyTorch | гибкий цикл обучения, богатая экосистема | препроцессинг живёт отдельно от модели, его легко рассинхронизировать |
| чтение картинок | **OpenCV (`cv2`)** | быстрый, сразу отдаёт `numpy`, grayscale одним аргументом | лишняя зависимость, читает BGR (для цветных картинок легко перепутать каналы) |
| | Pillow (`PIL`) | есть везде | медленнее на десятках тысяч файлов, нужен `np.array()` |
| | `tf.io.read_file` + `decode_png` | декодирование внутри графа, можно на GPU | сложнее отлаживать, выигрыш заметен только когда узкое место — ввод-вывод |

Выбраны Keras и OpenCV: 70 тысяч мелких файлов читаются один раз, тут важна
скорость чтения на CPU, а не декодирование в графе. Главный аргумент за Keras
именно в этой задаче — возможность положить нормализацию слоем внутрь модели
(подробно в разделе 8).

In [ ]:
import os                                     # обход каталогов: os.walk для поиска данных
from pathlib import Path                       # пути как объекты: удобнее строк, есть .stem и .suffix

import cv2                                     # чтение и ресайз изображений
import numpy as np                             # массивы: всё хранение данных
import pandas as pd                            # таблицы: чтение csv и сборка submission
import matplotlib.pyplot as plt                # графики
import seaborn as sns                          # тепловая карта confusion matrix
import tensorflow as tf                        # бэкенд: нужен для проверки GPU и политики точности
from tensorflow import keras                   # высокоуровневый API: модель, обучение, колбэки
from tensorflow.keras import layers            # слои: пишем layers.Conv2D вместо keras.layers.Conv2D

from sklearn.model_selection import train_test_split      # стратифицированное разбиение
from sklearn.metrics import classification_report          # precision/recall/f1 по каждому классу
from sklearn.metrics import confusion_matrix               # матрица «истина против предсказания»

print("tensorflow", tf.__version__)            # версии стоит печатать: API Keras 3 заметно отличается от 2.x
print("keras", keras.__version__)              # по этой строке потом понятно, почему код мог перестать работать

## 2. Конфигурация

### Какие были варианты

| Вариант | Плюсы | Минусы |
|---|---|---|
| **константы в одной ячейке сверху** | видно все ручки сразу, менять в одном месте | нужно помнить, что после правки ячейку надо перезапустить |
| числа прямо в коде по месту | писать быстрее | «магические числа»: чтобы поменять размер изображения, придётся искать его в пяти местах, и одно обязательно забудется |
| `argparse` / переменные окружения | удобно для скриптов и автотестов | в ноутбуке лишняя церемония |

Выбраны константы сверху. Отдельно про `IMG_SIZE = 64`: изображения в датасете
уже 64×64, и увеличивать их не нужно. Ресайз вверх не добавляет информации, но
умножает и объём памяти, и время обучения: 200×200 — это в 9.8 раза больше
пикселей на изображение и 2.6 ГиБ вместо 274 МиБ на весь датасет.

In [ ]:
SEED = 42                    # один сид на numpy, python и tensorflow: без него результат не повторить
IMG_SIZE = 64                # родное разрешение датасета; менять смысла нет, см. разбор выше
EPOCHS = 40                  # верхняя граница: реально обучение остановит EarlyStopping раньше
VAL_FRACTION = 0.10          # доля на валидацию: по ней принимаются решения об остановке
TEST_FRACTION = 0.10         # доля на тест: не участвует ни в обучении, ни в выборе модели
MAX_PER_CLASS = None         # None = брать все файлы; поставьте 50, чтобы прогнать пайплайн за минуту
USE_TTA = True               # усреднять предсказания по сдвигам изображения (раздел 17)

INPUT_DIR = Path("/kaggle/input")                                     # куда Kaggle монтирует данные
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")  # куда можно писать
SUBMISSION_PATH = WORK_DIR / "submission.csv"                          # файл, который заберёт лидерборд

keras.utils.set_random_seed(SEED)      # один вызов сеет random, numpy и tensorflow сразу
rng = np.random.default_rng(SEED)      # отдельный генератор для наших выборок: не зависит от вызовов Keras
sns.set_theme(style="whitegrid")       # единый стиль графиков, чтобы не настраивать каждый

GPUS = tf.config.list_physical_devices("GPU")     # список GPU; пустой, если ускоритель не включён
for gpu in GPUS:                                   # по умолчанию TF забирает всю память GPU сразу
    tf.config.experimental.set_memory_growth(gpu, True)   # просим выделять по мере надобности
if GPUS:                                           # на CPU половинная точность только замедлит
    keras.mixed_precision.set_global_policy("mixed_float16")  # веса float32, вычисления float16: ~2x быстрее
BATCH_SIZE = 256 if GPUS else 64                   # на GPU большой батч выгоден, на CPU он не влезет по памяти

print("GPU:", [g.name for g in GPUS] or "нет")                       # сразу видно, включён ли ускоритель
print("точность:", keras.mixed_precision.global_policy().name)        # проверка, что политика применилась
print("батч:", BATCH_SIZE, "| пишем в:", WORK_DIR)                    # куда попадёт submission

## 3. Поиск каталога с данными

### Какие были варианты

| Вариант | Плюсы | Минусы |
|---|---|---|
| хардкод `/kaggle/input/mashtots-dataset/Train` | одна строка | ломается при смене соревнования на v2, при вложенности `Train/Train` и при локальном запуске |
| `glob` по нескольким шаблонам | коротко | нужно заранее знать все возможные раскладки |
| **обход `os.walk` с признаком «папки с числовыми именами»** | работает при любой вложенности и любом названии соревнования | чуть больше кода |

Выбран обход. Признак искомого каталога — в нём лежит много подпапок с
числовыми именами (`0`, `1`, ... `77`). Это устойчиво: не зависит ни от имени
соревнования, ни от того, `Train/` там или `Train/Train/`.

Две тонкости в реализации ниже. Первая: спускаться внутрь самих папок-классов
незачем — там только картинки, а их там десятки тысяч, и обход будет долгим.
Вторая: ограничение по глубине не даёт уйти в бесконечный обход, если структура
окажется неожиданной.

In [ ]:
def find_class_root(min_classes: int = 10, max_depth: int = 4) -> Path:
    """Ищем каталог, внутри которого лежит min_classes и больше папок-классов."""
    # кандидаты: сначала каждое подключённое соревнование, потом локальные пути
    bases = sorted(p for p in INPUT_DIR.iterdir() if p.is_dir()) if INPUT_DIR.is_dir() else []
    bases += [Path("."), Path("data/mashtots")]        # чтобы ноутбук работал и вне Kaggle

    for base in bases:                                  # перебираем кандидатов по порядку
        if not base.is_dir():                           # локального пути может не быть — это нормально
            continue
        for dirpath, dirnames, _ in os.walk(base):      # dirnames можно менять на месте, os.walk это учтёт
            if len([d for d in dirnames if d.isdigit()]) >= min_classes:  # признак: много числовых подпапок
                return Path(dirpath)                    # нашли — дальше не идём
            if len(Path(dirpath).relative_to(base).parts) >= max_depth:   # слишком глубоко
                dirnames.clear()                        # пустой список = не спускаться ниже
            else:
                dirnames[:] = [d for d in dirnames if not d.isdigit()]    # в папки-классы не заходим

    raise FileNotFoundError(                            # понятная ошибка вместо падения дальше по коду
        "каталог с папками-классами не найден. В Kaggle нажмите Add Input и "
        "добавьте соревнование; локально положите данные в data/mashtots/Train"
    )


CLASS_ROOT = find_class_root()          # запоминаем: понадобится и для загрузки, и для поиска теста
print("данные:", CLASS_ROOT)            # печатаем, чтобы было видно, что нашлось именно то, что нужно

## 4. Ловушка первая: откуда брать номер класса

В Keras есть готовая функция `keras.utils.image_dataset_from_directory`, которая
сама обходит папки и назначает метки. Соблазнительно взять её и не писать
загрузчик руками. Но метки она назначает **по порядку сортировки имён папок**, а
имена у нас — числа в виде строк.

Строковая сортировка даёт `'0', '1', '10', '11', ..., '2', '20', ...`, поэтому
папка `10` получает метку 2, папка `77` — метку 5 (в примере ниже с шестью
папками). Модель обучится нормально, метрики будут хорошие, а в `submission.csv`
уедут чужие номера классов — и лидерборд покажет случайный результат при
полностью работающей модели.

Ячейка ниже показывает это на маленьком примере.

### Какие были варианты

| Вариант | Плюсы | Минусы |
|---|---|---|
| `image_dataset_from_directory` как есть | две строки кода, ленивая загрузка, не требует памяти | метки = порядок сортировки строк, а не имя папки; для числовых имён это молча неверно |
| она же, но с явным `class_names=[str(i) for i in range(78)]` | ловушка закрыта, ленивая загрузка сохраняется | всё равно нужно помнить про этот аргумент; порядок задаётся в другом месте, чем читается |
| **свой обход: метка = `int(имя папки)`** | метка берётся из имени папки, перепутать нечего; данные целиком в памяти, что упрощает разбиение и повторные прогоны | нужно написать ~20 строк; весь датасет должен влезать в RAM |

Выбран свой обход: 274 МиБ в `uint8` спокойно живут в памяти, а метка,
выведенная напрямую из имени папки, не зависит ни от порядка сортировки, ни от
версии Keras.

In [ ]:
# демонстрация на шести папках с числовыми именами: 0, 1, 2, 10, 11, 77
demo_root = Path("/tmp/label_order_demo")                    # временный каталог, к датасету отношения не имеет
for folder in ["0", "1", "2", "10", "11", "77"]:              # имена специально не по порядку
    (demo_root / folder).mkdir(parents=True, exist_ok=True)   # создаём папку-класс
    blank = np.zeros((8, 8, 1), dtype="uint8")                # содержимое неважно, нужны сами файлы
    keras.utils.save_img(demo_root / folder / "1.png", blank)  # одна картинка на класс

demo_ds = keras.utils.image_dataset_from_directory(           # тот самый удобный помощник
    demo_root, image_size=(8, 8), color_mode="grayscale",
    batch_size=1, shuffle=False, verbose=0,
)

print("порядок классов, который выбрал Keras:", demo_ds.class_names)   # строковая сортировка
for folder in ["10", "77"]:                                             # смотрим на два показательных случая
    print(f"  папка '{folder}' -> метка {demo_ds.class_names.index(folder)}"
          f"  (а по имени папки должно быть {folder})")

print("\nв нашем загрузчике метка берётся из имени папки:")
for folder in ["10", "77"]:
    print(f"  папка '{folder}' -> метка {int(folder)}")                  # int(имя) — и никаких сюрпризов

## 5. Чтение изображений в память

### Какие были варианты

| Что выбирать | Вариант | Почему так |
|---|---|---|
| тип массива | **`uint8`** | 274 МиБ на 70 060 изображений. Keras сам приведёт батч к `float32` при обучении |
| | `float32` сразу | 1.07 ГиБ — вчетверо больше памяти без единого плюса |
| выделение памяти | **`np.empty` заранее** | размер известен: сначала собираем список файлов, потом выделяем массив и заполняем |
| | `list.append` + `np.array(list)` | на пике в памяти лежат и список отдельных массивов, и его копия — примерно двойной расход |
| grayscale | **`cv2.imread(..., IMREAD_GRAYSCALE)`** | одна операция, сразу нужный формат |
| | `imread` + `cvtColor` | лишний проход по массиву; к тому же `cv2.IMREAD_GRAYSCALE` равен нулю и это **не** код цветового преобразования — подставив его в `cvtColor`, вы получите не то, что ожидали |
| ресайз | **только если размер отличается** | у 64×64 ресайз в 64×64 — это лишняя работа для 70 тысяч файлов |
| порядок файлов | **`sorted()`** | порядок `iterdir()` зависит от файловой системы; без сортировки один и тот же код даёт разные разбиения на разных машинах |
| битые файлы | **пропускать с подсчётом** | в датасетах попадаются нулевые файлы и служебные каталоги вроде `.ipynb_checkpoints`; падать из-за одного файла из 70 тысяч не стоит, но и молчать нельзя — поэтому счётчик выводится |

In [ ]:
IMAGE_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".pgm"}   # белый список: всё прочее не картинки


def list_class_dirs(root: Path) -> list[Path]:
    """Папки-классы, отсортированные численно: 0, 1, 2, ..., 10, ..., 77."""
    dirs = (p for p in root.iterdir() if p.is_dir() and p.name.isdigit())  # только числовые имена
    return sorted(dirs, key=lambda p: int(p.name))                         # ключ int, а не строка


def list_images(directory: Path) -> list[Path]:
    """Файлы-изображения внутри каталога, в стабильном порядке."""
    files = (p for p in directory.rglob("*") if p.is_file())               # rglob на случай вложенных папок
    return sorted(p for p in files if p.suffix.lower() in IMAGE_EXT)       # lower(): бывает и .PNG


def load_dataset(root: Path, max_per_class: int | None = None):
    """Возвращает X (N, 64, 64, 1) uint8 и y (N,) с метками из имён папок."""
    class_dirs = list_class_dirs(root)                     # список папок-классов
    samples = []                                            # пары (путь к файлу, метка)
    for class_dir in class_dirs:                            # идём по классам
        label = int(class_dir.name)                         # метка = имя папки, см. раздел 4
        files = list_images(class_dir)[:max_per_class]       # срез [:None] возвращает весь список
        samples += [(path, label) for path in files]         # добавляем все файлы этого класса

    X = np.empty((len(samples), IMG_SIZE, IMG_SIZE, 1), dtype=np.uint8)  # выделяем память один раз
    y = np.empty(len(samples), dtype=np.int16)               # int16 хватает на 78 классов
    progress_step = max(1, len(samples) // 5)                # печатать прогресс пять раз за проход

    kept = 0                                                 # сколько картинок реально прочиталось
    skipped = 0                                              # сколько файлов оказались нечитаемыми
    for i, (path, label) in enumerate(samples):              # enumerate нужен только для прогресса
        image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)  # cv2 не умеет Path, поэтому str()
        if image is None:                                    # cv2 не бросает исключение, а возвращает None
            skipped += 1
            continue                                         # битый файл просто пропускаем
        if image.shape != (IMG_SIZE, IMG_SIZE):              # ресайз только когда он действительно нужен
            image = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        X[kept, :, :, 0] = image                             # кладём в заранее выделенный массив
        y[kept] = label                                      # и метку туда же
        kept += 1                                            # счётчик = позиция следующей записи
        if (i + 1) % progress_step == 0:
            print(f"  прочитано {i + 1} из {len(samples)}")

    print(f"классов {len(class_dirs)} | загружено {kept} | пропущено {skipped}")
    return X[:kept], y[:kept]                                # обрезаем хвост, если что-то пропустили


X, y = load_dataset(CLASS_ROOT, MAX_PER_CLASS)               # основная загрузка
NUM_CLASSES = int(y.max()) + 1                                # число классов выводим из данных, не хардкодим
print(f"X: {X.shape} {X.dtype}, {X.nbytes / 2**20:.0f} МиБ | классов: {NUM_CLASSES}")

## 6. Посмотреть на данные глазами

Этот шаг легко пропустить, но он отвечает на вопросы, от которых зависят решения
дальше:

* **Какой фон?** Здесь фон чёрный (`0`), штрих светлый. Отсюда вывод для
  аугментации: пустоту после сдвига и поворота надо заливать нулями. Дефолт
  `fill_mode="reflect"` в слоях Keras отразил бы соседние пиксели и затащил в
  кадр обрывки штриха — модель училась бы на артефактах.
* **Сбалансированы ли классы?** Если отношение самого большого класса к самому
  маленькому близко к 1, `class_weight` не нужен. Если бы перекос был в разы,
  редкие классы модель бы игнорировала, и веса пришлось бы задать.
* **Правда ли изображения 64×64 и в градациях серого?** Если нет, вылезло бы в
  счётчике ресайза из предыдущей ячейки.

In [ ]:
counts = pd.Series(y).value_counts().sort_index()      # сколько изображений в каждом классе
ratio = counts.max() / counts.min()                     # мера перекоса классов
print(f"на класс: минимум {counts.min()}, медиана {int(counts.median())}, максимум {counts.max()}")
print(f"перекос {ratio:.2f} -> " + ("class_weight не нужен" if ratio < 1.5 else "стоит задать class_weight"))

fig, axes = plt.subplots(1, 2, figsize=(14, 3.2))       # два графика рядом: баланс и яркости
axes[0].bar(counts.index, counts.values, width=1.0)     # столбик на класс, width=1 чтобы не было щелей
axes[0].set(title="Изображений по классам", xlabel="класс")
axes[1].hist(X[:: max(1, len(X) // 500)].ravel(), bins=50)   # шаг по срезу: гистограмма по ~500 картинкам
axes[1].set(title="Значения пикселей", xlabel="значение", yscale="log")  # лог: нулей намного больше остального
plt.tight_layout()                                       # чтобы подписи не наезжали друг на друга
plt.show()

sample_idx = rng.choice(len(X), size=min(16, len(X)), replace=False)   # 16 случайных картинок без повторов
fig, axes = plt.subplots(2, 8, figsize=(13, 3.6))
for ax, i in zip(axes.ravel(), sample_idx):              # ravel() превращает сетку осей в плоский список
    ax.imshow(X[i, :, :, 0], cmap="gray")                # [i, :, :, 0] убирает ось канала: imshow ждёт 2D
    ax.set_title(f"класс {y[i]}", fontsize=8)
    ax.axis("off")                                       # оси у картинок только мешают
plt.tight_layout()
plt.show()

## 7. Разбиение на train / val / test

### Какие были варианты

| Вариант | Плюсы | Минусы |
|---|---|---|
| две части: train и test, тест же и в `validation_data` | проще, данных на обучение больше | по тесту принимаются решения (сколько эпох учить, какие веса сохранить), поэтому его оценка завышена — это утечка |
| **три части: train / val / test** | `val` для остановки обучения, `test` не видели ни разу — его оценка честная | на обучение уходит на 10 % меньше данных |
| K-fold кросс-валидация | самая надёжная оценка, а модели с фолдов можно усреднить в ансамбль | в K раз дольше; при 70 тысячах изображений одного разбиения достаточно |

Выбраны три части. `stratify=y` обязателен: он сохраняет пропорции всех 78
классов в каждой части. Без него редкий класс может целиком уехать в тест, и
тогда обучение его вообще не увидит.

Разбиение делается в два вызова: сначала отрезаем `val + test` от обучающей
части, потом делим этот кусок пополам. Поэтому во втором вызове доля считается
относительно уже отрезанного куска, а не всего датасета.

In [ ]:
X_train, X_hold, y_train, y_hold = train_test_split(     # первый разрез: train против (val + test)
    X, y,
    test_size=VAL_FRACTION + TEST_FRACTION,               # 0.20 = отложить пятую часть
    random_state=SEED,                                    # фиксируем, иначе разбиение будет разным каждый раз
    stratify=y,                                           # сохранить доли всех 78 классов
)

X_val, X_test, y_val, y_test = train_test_split(          # второй разрез: отложенное делим пополам
    X_hold, y_hold,
    test_size=TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION),   # доля внутри куска, а не всего датасета
    random_state=SEED,
    stratify=y_hold,                                      # стратификация нужна и здесь
)

del X_hold, y_hold                                        # промежуточный кусок больше не нужен, освобождаем память

for name, images, labels in [("train", X_train, y_train),  # печатаем размеры: заметно, если что-то пошло не так
                             ("val", X_val, y_val),
                             ("test", X_test, y_test)]:
    print(f"{name:<6} {len(images):>7} изображений | классов {len(np.unique(labels))}")

## 8. Нормализация: важно не «как», а «где»

Пиксели приходят целыми числами `0..255`. Без нормализации сеть почти
гарантированно не обучится: после `Flatten` предактивации получаются порядка
десятков тысяч, первый же шаг Adam выбрасывает почти все ReLU в отрицательную
область, а ReLU при отрицательном входе даёт нулевой градиент. Дальше сеть
способна выдавать только константу, и лучшая константа для кросс-энтропии —
равномерное распределение, то есть `loss = ln(78) = 4.3567` и
`accuracy = 1/78 = 0.0128`.

Делить на 255 умеет каждый. Вопрос в том, в каком месте это делать.

| Где нормализовать | Плюсы | Минусы |
|---|---|---|
| в `numpy` перед обучением: `X = X / 255.0` | очевидно и наглядно | массив становится `float32` — 1.07 ГиБ вместо 274 МиБ; и главное, при предсказании на новой картинке про деление легко забыть, а модель тогда выдаёт мусор без всякой ошибки |
| в конвейере `tf.data.map` | памяти не жрёт, работает лениво | конвейер для обучения и код для одиночного предсказания — это два разных места, которые надо держать согласованными руками |
| **слоем `Rescaling` внутри модели** | нормализация — часть модели: она применяется и при обучении, и при `predict`, и после `model.save()`/загрузки. Забыть её невозможно | на входе модели нужно помнить, что она ждёт `0..255`, а не `0..1` |

Выбран слой внутри модели. Расхождение препроцессинга между обучением и
инференсом — одна из самых частых и самых незаметных ошибок в ML-коде, потому
что она не вызывает исключения, а просто портит предсказания. Слой внутри
модели убирает саму возможность такой ошибки.

Отдельный вопрос — шкала: `[0, 1]` делением на 255 или стандартизация
`(x - mean) / std`. Для сети с BatchNorm разница невелика: первый же BN всё
равно приведёт активации к своей шкале. Выбрано деление на 255 как более
простое и не требующее хранить статистики датасета.

## 9. Аугментация

Аугментация — случайные искажения обучающих картинок. Смысл: модель видит
каждую букву в слегка разных вариантах и учится узнавать саму форму, а не
запоминать конкретные пиксели.

### Какие были варианты инструмента

| Вариант | Плюсы | Минусы |
|---|---|---|
| **слои `RandomRotation` / `RandomTranslation` / `RandomZoom`** | часть модели, считаются на GPU, автоматически отключаются вне обучения (в `predict` они no-op) | набор преобразований скромнее, чем у специализированных библиотек |
| `ImageDataGenerator` | много примеров в старых руководствах | в Keras 3 устаревший API, работает на CPU и часто становится узким местом, GPU простаивает |
| `albumentations` | самый богатый набор преобразований | работает на CPU, внешняя зависимость, препроцессинг снова отделяется от модели |

### Какие были варианты состава преобразований

| Преобразование | Взято? | Почему |
|---|---|---|
| поворот ±10.8°, сдвиг ±8 %, зум ±10 % | да | рукописные буквы естественно варьируются по наклону, положению и размеру — это те искажения, которые встретятся и в тесте |
| горизонтальное отражение | **нет** | буквы зеркально несимметричны. Отражённая буква — это либо другая буква, либо не буква вовсе. Такая аугментация не расширяет данные, а добавляет неверные метки |
| `fill_mode="constant", fill_value=0` | да | фон в датасете чёрный, поэтому пустота после поворота должна быть чёрной |
| `fill_mode="reflect"` (дефолт) | нет | отразил бы край и затащил в кадр обрывки штриха — модель училась бы на артефактах |

Амплитуды намеренно небольшие. При слишком сильных искажениях буквы начинают
переходить друг в друга (особенно похожие пары), и аугментация начинает мешать,
а не помогать.

## 10. Архитектура сети

### Своя CNN или готовая предобученная?

| Вариант | Плюсы | Минусы |
|---|---|---|
| **своя небольшая CNN (~2.5 М параметров)** | обучается с нуля за минуты, ровно под 64×64 и один канал | нужно самому выбрать структуру |
| transfer learning (ResNet50, EfficientNet) | мощные признаки, обычно выигрывает на маленьких датасетах | предобучены на цветных фотографиях 224×224. Нашу картинку пришлось бы растянуть в 12 раз по площади и размножить в 3 канала; домен рукописного штриха от фотографий далёк, а считать в 20+ раз дороже |
| ResNet с нуля | глубина окупается на больших данных | при 70 тысячах картинок 64×64 выигрыш небольшой, а времени и риска больше |

Выбрана своя CNN: 70 тысяч изображений — достаточно, чтобы обучить сеть с нуля,
а предобученные признаки из мира фотографий здесь дают мало.

### Два `Conv 3×3` вместо одного `Conv 5×5`

Две свёртки 3×3 подряд «видят» ту же область 5×5, что и одна 5×5, но дешевле и
выразительнее. Для перехода 32 → 32 канала: одна `Conv 5×5` — это 25 632
параметра, две `Conv 3×3` — 18 496. Плюс между ними стоит нелинейность, то есть
на том же поле обзора сеть может выразить более сложную функцию.

### `Flatten` или `GlobalAveragePooling`

| Вариант | Параметров в голове | Смысл |
|---|---|---|
| **`Flatten` → `Dense(256)`** | 2 097 408 | сохраняет информацию о том, **где** находится признак |
| `GlobalAveragePooling` → `Dense(256)` | 33 024 | усредняет каждый канал по всей картинке, положение теряется |

Разница в 63 раза по числу параметров, и обычно это аргумент за пулинг. Но здесь
выбран `Flatten`: буквы различаются именно взаимным расположением штрихов, а
`GlobalAveragePooling` эту информацию выбрасывает. Если бы датасет был
существенно меньше и сеть переобучалась, стоило бы переключиться на пулинг.

### Ловушка вторая: узкое место в голове

Если между свёртками и выходным слоем поставить очень узкий слой, например
`Dense(10, relu)` → `Dense(2, relu)` → `Dense(78, softmax)`, сеть не обучится
вообще. Формально softmax поверх двумерного признака может нарезать плоскость на
78 выпуклых областей, так что «математически невозможно» — неверная
формулировка. Практически же обучить такое представление нельзя: градиент идёт
через два подряд сужающихся ReLU, и как только один из двух последних нейронов
уходит в отрицательную область, половина представления обнуляется навсегда.
Дальше — та же константа и тот же `loss = ln(78)`. Раздел 12 показывает это
запуском.

Отсюда правило: **между последним свёрточным блоком и выходом не должно быть
слоёв уже, чем нужно для представления числа классов.** Здесь голова
`Flatten → Dense(256) → Dense(78)` — сужений нет.

### Нормализация активаций

| Вариант | Замер на этой задаче (6 эпох, малая выборка) |
|---|---|
| **BatchNorm, `momentum=0.9`** | train 0.920, val **0.999**, val-loss 0.15 |
| BatchNorm, `momentum=0.99` (дефолт) | train 0.920, val **0.018**, val-loss вырос до 9.01 |
| GroupNorm | train 0.584, val 0.973 |
| без нормализации | train 0.681, val 0.981 |

Про `momentum` подробно: на инференсе BatchNorm использует не статистики
текущего батча, а накопленные скользящие средние, которые обновляются как
`m ← momentum·m + (1 − momentum)·батч`. После `k` шагов от инициализации
(`mean=0, var=1`) остаётся доля `momentum^k`. При `momentum=0.99` даже через
400 шагов это ещё 1.8 %, а сходится только к ~2000 шагам; при `0.9` хватает
сотни шагов (`0.9^100 ≈ 3·10⁻⁵`). Пока статистики не сошлись, обучение идёт
нормально, а валидация показывает случайное угадывание — очень легко принять
это за «модель не учится» и начать чинить не то.

### Почему выходной слой явно `dtype="float32"`

При `mixed_float16` слои считают в половинной точности. Для softmax это опасно:
экспонента легко уходит в переполнение, а сумма — в потерю точности. Поэтому
последний слой принудительно возвращается во `float32`.

In [ ]:
BN_MOMENTUM = 0.9      # см. разбор выше: с дефолтным 0.99 валидация врёт первые тысячи шагов


def build_model(num_classes: int = NUM_CLASSES) -> keras.Model:
    """Функциональный API: явный граф, легко вставить слои препроцессинга в начало."""
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="image")   # модель ждёт uint8 0..255

    # --- препроцессинг внутри модели ---
    x = layers.Rescaling(1.0 / 255)(inputs)          # 0..255 -> 0..1; работает и при predict, см. раздел 8
    x = layers.RandomRotation(                        # случайный поворот
        0.03,                                         # фактор задаётся долей от 360°, поэтому это ±10.8°
        fill_mode="constant", fill_value=0.0,         # пустоту заливаем чёрным, как фон датасета
    )(x)
    x = layers.RandomTranslation(0.08, 0.08, fill_mode="constant", fill_value=0.0)(x)  # сдвиг ±8 % по обеим осям
    x = layers.RandomZoom(0.10, fill_mode="constant", fill_value=0.0)(x)               # масштаб ±10 %
    # горизонтального отражения нет намеренно: буквы зеркально несимметричны

    # --- свёрточная часть: три блока, каналы 32 -> 64 -> 128 ---
    for filters in (32, 64, 128):                     # с каждым блоком картинка меньше, а каналов больше
        for _ in range(2):                            # две свёртки 3x3 = поле обзора 5x5, но дешевле
            x = layers.Conv2D(
                filters, 3,
                padding="same",                       # "same" сохраняет размер: уменьшает только MaxPooling
                use_bias=False,                       # сразу за свёрткой идёт BN, он всё равно вычтет среднее
            )(x)
            x = layers.BatchNormalization(momentum=BN_MOMENTUM)(x)   # стабилизирует обучение
            x = layers.Activation("relu")(x)          # нелинейность отдельным слоем: BN должен идти до неё
        x = layers.MaxPooling2D(2)(x)                 # 64 -> 32 -> 16 -> 8 по стороне
        x = layers.Dropout(0.25)(x)                   # регуляризация: гасим часть карт признаков

    # --- голова: из карт признаков в вероятности классов ---
    x = layers.Flatten()(x)                           # 8*8*128 = 8192 признака; сохраняет положение, см. разбор
    x = layers.Dense(256, use_bias=False)(x)          # снова без bias, потому что дальше BN
    x = layers.BatchNormalization(momentum=BN_MOMENTUM)(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.40)(x)                       # в голове параметров больше всего, поэтому dropout выше
    outputs = layers.Dense(
        num_classes,                                   # ровно столько выходов, сколько классов
        activation="softmax",                          # вероятности, сумма по классам = 1
        dtype="float32",                               # softmax во float16 неустойчив, см. разбор
        name="probs",                                  # имя пригодится, если понадобится вытащить слой
    )(x)

    return keras.Model(inputs, outputs, name="mashtots_cnn")

## 11. Компиляция: функция потерь, оптимизатор, метрики

### Функция потерь

| Вариант | Плюсы | Минусы |
|---|---|---|
| **`sparse_categorical_crossentropy`** | метки — просто целые числа. Не нужна матрица `N×78` и не нужно нигде хардкодить число классов | нельзя задать `label_smoothing` |
| `categorical_crossentropy` + `to_categorical` | поддерживает `label_smoothing` | лишняя матрица в памяти, а `to_categorical(y, num_classes=78)` требует указать 78 руками — легко разойтись с данными |

Выбран `sparse`. Если понадобится `label_smoothing` (в рукописном тексте есть
объективно неоднозначные образцы, и смягчение метки снижает переуверенность
модели), придётся перейти на `CategoricalCrossentropy` с one-hot метками.

### Оптимизатор

| Вариант | Плюсы | Минусы |
|---|---|---|
| **Adam, `lr=1e-3`** | работает из коробки почти на любой задаче, сам подстраивает шаг для каждого параметра | итоговое качество иногда чуть ниже, чем у хорошо настроенного SGD |
| SGD + momentum + расписание | при аккуратной настройке часто даёт лучший результат | нужно подбирать и шаг, и расписание — это отдельная работа |
| AdamW | корректная L2-регуляризация | ещё один гиперпараметр; здесь регуляризацию уже дают dropout и аугментация |

### Метрики

`accuracy` — то, что в итоге спрашивает соревнование. `top-3` добавлена как
диагностика: если accuracy низкая, а top-3 высокая, значит сеть выделяет
правильную букву в число похожих, но не может выбрать между несколькими
близкими начертаниями — это совсем другая проблема, чем «сеть не учится».

In [ ]:
def compile_model(model: keras.Model) -> keras.Model:
    """Собран отдельной функцией, чтобы обучение и переобучение шли одинаково."""
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),                 # 1e-3 — стандартная стартовая скорость для Adam
        loss="sparse_categorical_crossentropy",                 # метки целые, one-hot не нужен
        metrics=[
            "accuracy",                                         # основная метрика соревнования
            keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3"),  # диагностическая, см. разбор
        ],
    )
    return model


model = compile_model(build_model())     # создаём и сразу компилируем
model.summary()                           # таблица слоёв: проверьте, что параметров ~2.5 М и нет узких Dense

## 12. Демонстрация: что делает узкое место в голове

Сравниваем три конфигурации на одной и той же подвыборке и с одной и той же
инициализацией. Различаются только две вещи: нормализуется ли вход и есть ли
`Dense(2)` перед выходом. Всё остальное одинаково.

Ориентиры, с которыми надо сравнивать результат: `ln(78) = 4.3567` — это
кросс-энтропия равномерного распределения, `1/78 = 0.0128` — accuracy случайного
угадывания. Если сеть показывает эти числа, она не учится вообще.

Ячейка занимает пару минут. Она нужна только для понимания и на итог не влияет —
можно пропустить.

In [ ]:
def demo_model(normalize: bool, bottleneck: bool) -> keras.Model:
    """Упрощённая сеть с двумя переключателями — ровно два исследуемых фактора."""
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 1))
    x = layers.Rescaling(1.0 / 255)(inp) if normalize else inp   # без нормализации вход остаётся 0..255
    x = layers.Conv2D(32, 3, activation="relu")(x)                # без BN: чтобы он не спас положение
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(32, activation="relu")(x)
    if bottleneck:                                                # тот самый вредный участок
        x = layers.Dense(10, activation="relu")(x)
        x = layers.Dense(2, activation="relu", name="bottleneck")(x)   # 78 классов через 2 нейрона
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return keras.Model(inp, out)


demo_idx = rng.choice(len(X), size=min(4096, len(X)), replace=False)   # подвыборка, чтобы было быстро
X_demo, y_demo = X[demo_idx], y[demo_idx].astype("int32")

for title, normalize, bottleneck in [
    ("вход 0..255, есть Dense(2)  ", False, True),
    ("нормализован, есть Dense(2) ", True, True),
    ("нормализован, без Dense(2)  ", True, False),
]:
    keras.utils.set_random_seed(SEED)                    # одинаковая инициализация у всех трёх
    demo = demo_model(normalize, bottleneck)
    demo.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    demo.fit(X_demo, y_demo, epochs=5, batch_size=32, verbose=0)       # verbose=0: нас интересует только итог
    loss, acc = demo.evaluate(X_demo, y_demo, verbose=0)

    note = ""
    if bottleneck:                                        # заглянем внутрь узкого слоя
        probe = keras.Model(demo.inputs, demo.get_layer("bottleneck").output)  # модель до нужного слоя
        zeros = np.asarray(probe.predict(X_demo, verbose=0)) == 0               # ReLU вернул ровно ноль
        note = (f" | нулей на выходе Dense(2): {zeros.mean():>5.1%}"
                f", мёртвых нейронов: {int(zeros.all(axis=0).sum())} из 2")     # мёртвый = ноль на всех входах
    print(f"{title} loss={loss:.4f} accuracy={acc:.4f}{note}")

print(f"\nориентиры: ln(78)={np.log(NUM_CLASSES):.4f}, 1/78={1 / NUM_CLASSES:.4f}")

## 13. Колбэки: чем закончить обучение

Фиксированное число эпох — плохой вариант: угадать его заранее нельзя, и вы
либо не доучите, либо переобучите. Колбэки решают это за вас.

| Колбэк | Что делает | Альтернатива и почему не она |
|---|---|---|
| `EarlyStopping(restore_best_weights=True)` | останавливает, когда `val_accuracy` перестала расти, и **возвращает лучшие веса**, а не последние | без `restore_best_weights` в модели останутся веса последней эпохи — обычно уже переобученные |
| `ReduceLROnPlateau` | делит скорость обучения на 2 на плато | `CosineDecay` даёт расписание заранее, без оглядки на метрики; он часто чуть лучше, но требует заранее знать число эпох, а у нас его определяет early stopping |
| `ModelCheckpoint(save_best_only=True)` | пишет лучшую модель в файл | частично дублирует `restore_best_weights`, но даёт файл на диске: если сессия упадёт, обученная модель не потеряется |

Важно про `patience`: у `EarlyStopping` он должен быть заметно больше, чем у
`ReduceLROnPlateau`. Иначе обучение остановится раньше, чем понижение скорости
успеет дать эффект. Здесь 8 против 3.

Ещё одна деталь: `EarlyStopping` следит за `val_accuracy`, а `ReduceLROnPlateau`
за `val_loss`. Это не непоследовательность. Accuracy — то, что оценивает
соревнование, поэтому останавливаемся по ней. А loss меняется плавнее и раньше
показывает выход на плато, поэтому решение о понижении скорости лучше принимать
по нему.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",          # останавливаемся по метрике соревнования
        patience=8,                       # ждём 8 эпох без улучшения, прежде чем остановиться
        restore_best_weights=True,        # вернуть веса лучшей эпохи, а не последней
        verbose=1,                        # напечатать, на какой эпохе остановились
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",               # loss реагирует раньше и плавнее, чем accuracy
        factor=0.5,                       # на плато делим скорость обучения вдвое
        patience=3,                       # заметно меньше, чем у EarlyStopping, см. разбор
        min_lr=1e-5,                      # ниже этого понижать бессмысленно
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        WORK_DIR / "mashtots_cnn.keras",  # формат .keras — актуальный для Keras 3
        monitor="val_accuracy",
        save_best_only=True,              # перезаписывать файл только при улучшении
    ),
]

## 14. Обучение

### Чем кормить `fit`: numpy или `tf.data`

| Вариант | Плюсы | Минусы |
|---|---|---|
| **массивы numpy напрямую** | одна строка, Keras сам нарежет батчи и перемешает | весь датасет должен быть в памяти |
| `tf.data.Dataset` | ленивая загрузка, `prefetch`, работает с данными любого размера | больше кода; `from_tensor_slices` на большом массиве всё равно скопирует его целиком |

Данные уже в памяти и занимают 274 МиБ, поэтому выбран простой вариант.
`tf.data` понадобился бы, если бы датасет не влезал в RAM.

Про `verbose=2`: это одна строка на эпоху. При `verbose=1` в ноутбук пишется
анимированный прогресс-бар, и сохранённый `.ipynb` раздувается до десятков
мегабайт — на Kaggle это заметная проблема.

In [ ]:
history = model.fit(
    X_train, y_train,                        # массивы numpy: Keras сам нарежет батчи
    validation_data=(X_val, y_val),           # по этой части работают EarlyStopping и ReduceLROnPlateau
    epochs=EPOCHS,                            # верхняя граница, реально остановит EarlyStopping
    batch_size=BATCH_SIZE,                    # 256 на GPU, 64 на CPU
    callbacks=callbacks,                      # см. предыдущий раздел
    verbose=2,                                # одна строка на эпоху вместо прогресс-бара
)

## 15. Кривые обучения и как их читать

Четыре типичные картины и что каждая означает:

| train | val | Диагноз |
|---|---|---|
| стоит на `1/78` | стоит на `1/78` | сеть не учится: ненормализованный вход, узкое место в голове, слишком большая скорость обучения или перепутанные метки |
| растёт | стоит на `1/78`, val-loss **растёт** | слои, по-разному работающие в обучении и инференсе — почти всегда BatchNorm с несошедшимися статистиками (раздел 10) |
| растёт | растёт, но заметно ниже train | обычное переобучение: усилить аугментацию и dropout, добавить данных |
| растёт | растёт вместе с train | всё в порядке |

Красные пунктирные линии на графиках — уровень «сеть не учится». Полезно
держать их перед глазами: без ориентира легко посчитать `accuracy = 0.013`
просто плохим результатом, а не признаком сломанного пайплайна.

In [ ]:
hist = pd.DataFrame(history.history)          # словарь истории удобнее смотреть как таблицу

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
hist[["loss", "val_loss"]].plot(ax=axes[0], title="Loss")               # две кривые на одних осях
axes[0].axhline(np.log(NUM_CLASSES), ls="--", c="r",                     # ориентир «равномерный выход»
                label=f"ln({NUM_CLASSES}) — сеть не учится")
hist[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy")
axes[1].axhline(1 / NUM_CLASSES, ls="--", c="r",                         # ориентир «случайное угадывание»
                label=f"1/{NUM_CLASSES} — случайное угадывание")
for ax in axes:
    ax.set_xlabel("эпоха")
    ax.legend()
plt.tight_layout()
plt.show()

best_epoch = int(np.argmax(history.history["val_accuracy"])) + 1   # +1: эпохи нумеруются с единицы
print(f"лучшая эпоха {best_epoch} из {len(hist)} пройденных")

## 16. Оценка на отложенном тесте

`X_test` до этого момента не участвовал ни в обучении, ни в решениях о нём —
поэтому эти числа честные, в отличие от валидационных.

### Зачем несколько метрик, а не одна accuracy

* **accuracy** — то, что оценивает соревнование, но она ничего не говорит о том,
  *где* модель ошибается.
* **top-3** — если она сильно выше accuracy, модель уверенно сужает выбор до
  нескольких похожих начертаний, но не различает их. Это подсказывает, куда
  смотреть дальше.
* **отчёт по классам** — показывает конкретные классы, которые модель не выучила.
  Средняя accuracy может быть высокой, а несколько классов при этом полностью
  проваленными.
* **confusion matrix** — показывает, *с чем именно* путается каждый класс. В этой
  задаче ожидаемо путаются заглавные и строчные варианты одной буквы; для них
  может помочь двухступенчатая схема (сначала буква, потом регистр).

In [ ]:
scores = model.evaluate(X_test, y_test, batch_size=BATCH_SIZE, verbose=0, return_dict=True)
for name, value in scores.items():            # return_dict=True даёт имена метрик
    print(f"test {name:<10} {value:.4f}")     # без него пришлось бы угадывать порядок чисел

probs = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)   # вероятности: (N, 78)
y_pred = probs.argmax(axis=1)                                      # класс = индекс максимума по строке

report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)  # zero_division: без warning
per_class = pd.DataFrame(report).T                                  # транспонируем: классы станут строками
per_class = per_class.loc[per_class.index.str.isdigit()]             # убираем итоговые строки accuracy/avg
per_class = per_class.astype({"support": int}).sort_values("f1-score")   # худшие классы наверх
print("\n10 самых трудных классов:")
print(per_class.head(10).round(3))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=np.arange(NUM_CLASSES))   # labels: чтобы попали все 78 классов

plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap="viridis", square=True, cbar_kws={"shrink": 0.7})  # 78x78 числами не подписать
plt.title("Confusion matrix")
plt.xlabel("предсказано")
plt.ylabel("истина")
plt.tight_layout()
plt.show()

off = cm.copy()                            # копия: диагональ портить нельзя, cm ещё нужен
np.fill_diagonal(off, 0)                    # диагональ — это правильные ответы, они не интересны
flat_order = np.argsort(off, axis=None)[::-1]                        # индексы по убыванию, но плоские
pairs = np.dstack(np.unravel_index(flat_order, off.shape))[0][:10]   # обратно в пары (строка, столбец)
pairs = [(a, b) for a, b in pairs if off[a, b] > 0]                   # нули отбрасываем

if not pairs:
    print("перепутанных пар нет")
else:
    print("Чаще всего путаются (истина -> предсказание, случаев):")
    for a, b in pairs:
        print(f"  {a:>2} -> {b:>2} : {off[a, b]}")

## 17. TTA — усреднение предсказаний по сдвигам

Test-time augmentation: предсказываем несколько раз, слегка сдвигая картинку, и
усредняем вероятности. Обычно даёт небольшой, но почти бесплатный прирост —
одиночная ошибка на конкретном сдвиге усредняется с четырьмя другими взглядами.

| Вариант | Плюсы | Минусы |
|---|---|---|
| без TTA | быстрее в 5 раз | теряется небольшой прирост |
| **`np.roll` на ±2 пикселя** | полностью детерминированно, результат повторяется от запуска к запуску | сдвиг циклический: то, что уходит за край, появляется с другой стороны. Здесь это безвредно, потому что края чёрные |
| слои `Random*` с `training=True` | те же преобразования, что при обучении | случайно: два запуска дадут разные предсказания, и расхождение с лидербордом не объяснить |

Отражения в TTA не используются намеренно — по той же причине, что и в
аугментации: буквы зеркально несимметричны.

In [ ]:
def predict_with_tta(model: keras.Model, images: np.ndarray, use_tta: bool = True) -> np.ndarray:
    """Средние вероятности по нескольким сдвигам изображения."""
    shifts = [(0, 0), (0, 2), (0, -2), (2, 0), (-2, 0)] if use_tta else [(0, 0)]  # центр плюс четыре сдвига
    total = np.zeros((len(images), NUM_CLASSES), dtype=np.float32)                 # накопитель сумм
    for dy, dx in shifts:
        # axis=(1, 2) — это высота и ширина; ось 0 это номер картинки, ось 3 — канал
        batch = images if (dy, dx) == (0, 0) else np.roll(images, (dy, dx), axis=(1, 2))
        total += model.predict(batch, batch_size=BATCH_SIZE, verbose=0)             # суммируем вероятности
    return total / len(shifts)                                                       # среднее

## 18. Ловушка третья: порядок строк в submission

Самая обидная из трёх, потому что модель при ней полностью рабочая, а
лидерборд показывает случайный результат.

Файлы читаются в порядке `sorted()`, то есть **лексикографически**:
`1.png, 10.png, 100.png, 11.png, 2.png`. А `sample_submission.csv` обычно
отсортирован численно: `1, 2, 10, 11, 100`. Если просто приписать предсказания
к строкам `sample_submission` по порядку, предсказание для файла `10.png`
окажется в строке идентификатора `2`.

| Вариант | Плюсы | Минусы |
|---|---|---|
| позиционно: `pd.DataFrame({"Id": ids, "Category": preds})` | одна строка | верно только если порядок чтения файлов совпал с порядком в `sample_submission`. Проверить это глазами почти невозможно, а ошибка не вызывает исключения |
| численная сортировка имён файлов | закрывает конкретно этот случай | а если идентификаторы не числа, а `img_0007`, всё снова разъедется |
| **сопоставление по идентификатору через словарь** | порядок чтения перестаёт иметь значение | нужно учесть, что идентификатор может быть записан как `123` или как `123.png` |

Выбран словарь. В него класс кладётся под обоими вариантами ключа — и `123`,
и `123.png`, — а потом идентификаторы из `sample_submission` просто ищутся в
нём. Функция печатает, сколько идентификаторов удалось сопоставить: если
сопоставилось не всё, это видно сразу, а не после отправки.

Ячейка ниже показывает расхождение порядков на пяти именах.

In [ ]:
demo_files = sorted([f"{i}.png" for i in [1, 2, 10, 11, 100]])   # порядок, в котором файлы прочитаются
demo_sample_ids = ["1", "2", "10", "11", "100"]                   # порядок строк в sample_submission
demo_preds = [f"класс файла {name}" for name in demo_files]        # предсказания идут в порядке чтения

print("порядок чтения файлов :", demo_files)
print("порядок в sample      :", demo_sample_ids)
print("\nесли приписать предсказания по позиции:")
for sample_id, prediction in zip(demo_sample_ids, demo_preds):     # zip по позиции — так делать нельзя
    mark = "верно" if prediction.endswith(f"{sample_id}.png") else "ОШИБКА"
    print(f"  строка Id={sample_id:>3}  получит  {prediction:<22} -> {mark}")

In [ ]:
def stack_images(paths: list[Path]) -> tuple[np.ndarray, list[Path]]:
    """Читает файлы в один массив; возвращает и список фактически прочитанных путей."""
    out = np.empty((len(paths), IMG_SIZE, IMG_SIZE, 1), dtype=np.uint8)   # память под максимум
    kept: list[Path] = []                                                  # что удалось прочитать
    for path in paths:
        image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)                # тот же способ, что при обучении
        if image is None:                                                   # битый файл пропускаем
            continue
        if image.shape != (IMG_SIZE, IMG_SIZE):
            image = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        out[len(kept), :, :, 0] = image                                     # позиция = сколько уже сохранили
        kept.append(path)                                                   # путь нужен, чтобы взять из него id
    return out[:len(kept)], kept                                            # обрезаем неиспользованный хвост


def find_competition_test(prefer: Path | None = None):
    """Тест соревнования: каталог картинок или таблица с пикселями. None, если не найден."""
    comps = sorted(p for p in INPUT_DIR.iterdir() if p.is_dir()) if INPUT_DIR.is_dir() else [Path(".")]
    if prefer is not None:                                # если подключено несколько соревнований,
        comps = [prefer] + [c for c in comps if c != prefer]   # тест берём из того же, откуда train

    for comp in comps:
        sample = next(iter(sorted(comp.rglob("sample_submission.csv"))), None)   # None, если файла нет

        for name in ("new_test", "Test", "test"):          # вариант первый: каталог с картинками
            for directory in sorted(comp.rglob(name)):
                if not directory.is_dir():
                    continue
                files = list_images(directory)
                if files:
                    images, kept = stack_images(files)
                    return images, [p.name for p in kept], sample   # id = имя файла, например 123.png

        for csv_path in sorted(comp.rglob("new_test.csv")):  # вариант второй: таблица с пикселями
            table = pd.read_csv(csv_path)
            numeric = table.select_dtypes(include="number")            # текстовые колонки не пиксели
            pixel_cols = [c for c in numeric.columns if numeric[c].between(0, 255).all()]
            if len(pixel_cols) >= IMG_SIZE * IMG_SIZE:                  # 64*64 = 4096 колонок и больше
                pixels = numeric[pixel_cols[-IMG_SIZE * IMG_SIZE:]]     # последние 4096: первой обычно идёт id
                images = pixels.to_numpy(dtype=np.uint8).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
                ids = table[table.columns[0]].astype(str).tolist()      # первая колонка — идентификатор
                return images, ids, sample
    return None


COMPETITION_DIR = next((p for p in [CLASS_ROOT, *CLASS_ROOT.parents] if p.parent == INPUT_DIR), None)
test_data = find_competition_test(prefer=COMPETITION_DIR)   # ищем тест в том же соревновании, что и train

if test_data is None:
    print("тест соревнования не найден — submission не создаётся")
else:
    test_images, test_ids, sample_path = test_data
    print(f"тестовых изображений {len(test_images)} | sample_submission: {sample_path}")
    test_probs = predict_with_tta(model, test_images, USE_TTA)          # см. раздел 17
    test_pred = test_probs.argmax(axis=1)                               # класс = индекс максимума
    print(f"средняя уверенность {test_probs.max(axis=1).mean():.3f}")   # низкая говорит о слабой модели

In [ ]:
def build_submission(ids, preds, sample_path) -> pd.DataFrame:
    """Раскладывает предсказания по идентификаторам из sample_submission."""
    by_id: dict[str, int] = {}                        # словарь: идентификатор -> предсказанный класс
    for raw_id, cls in zip(ids, preds):                # zip здесь корректен: оба списка в порядке чтения
        key = str(raw_id)
        by_id[key] = int(cls)                          # вариант ключа как есть, например 123.png
        by_id[Path(key).stem] = int(cls)               # и без расширения, например 123

    if sample_path is None or not Path(sample_path).is_file():   # sample_submission может отсутствовать
        print("sample_submission.csv нет — пишем в порядке файлов, колонки Id/Category")
        return pd.DataFrame({"Id": [str(i) for i in ids], "Category": [int(p) for p in preds]})

    sample = pd.read_csv(sample_path)                  # берём формат из самого соревнования
    id_col = sample.columns[0]                          # имена колонок не угадываем, а читаем
    target_col = sample.columns[1] if len(sample.columns) > 1 else "Category"

    keys = sample[id_col].astype(str)                                   # astype(str): id могут быть числами
    matched = keys.map(lambda k: by_id.get(k, by_id.get(Path(k).stem)))  # пробуем оба варианта ключа
    missing = int(matched.isna().sum())                                  # сколько не нашлось

    if missing == len(sample):                          # ни один не совпал — формат id совсем другой
        print("ни один идентификатор не совпал; пишем в порядке файлов, проверьте формат вручную")
        return pd.DataFrame({id_col: [str(i) for i in ids], target_col: [int(p) for p in preds]})
    if missing:                                          # частичное несовпадение тоже надо видеть
        print(f"не нашлось предсказаний для {missing} из {len(sample)} идентификаторов")
    print(f"сопоставлено {len(sample) - missing} из {len(sample)} идентификаторов")

    out = sample.copy()                                  # сохраняем порядок строк соревнования
    out[target_col] = matched.fillna(0).astype(int)       # fillna: пропуски недопустимы в submission
    return out[[id_col, target_col]]                      # только нужные колонки, в нужном порядке


if test_data is not None:
    submission = build_submission(test_ids, test_pred, sample_path)
    submission.to_csv(SUBMISSION_PATH, index=False)       # index=False: лишняя колонка сломает формат
    print(f"\n{SUBMISSION_PATH}: строк {len(submission)}, колонки {list(submission.columns)}")
    display(submission.head())

## 19. Проверка файла перед отправкой

Число попыток в соревновании ограничено, поэтому дешевле проверить файл здесь.
Проверяем то, что чаще всего ломается: количество строк, уникальность
идентификаторов, отсутствие пропусков, диапазон классов и совпадение множества
идентификаторов с `sample_submission`.

Отдельно стоит смотреть на число различных предсказанных классов. Если модель
на 78 классов предсказала, скажем, три, то формально файл корректен, а
фактически модель не работает.

In [ ]:
if not SUBMISSION_PATH.is_file():
    print("submission.csv не создан")
else:
    sub = pd.read_csv(SUBMISSION_PATH)                       # читаем с диска, а не из переменной:
    id_col, target_col = sub.columns[0], sub.columns[1]       # так проверяется то, что реально отправится

    checks = {
        "строк": len(sub),
        "идентификаторы уникальны": bool(sub[id_col].is_unique),          # дубли ломают оценку
        "пропусков нет": bool(sub.notna().all().all()),                    # NaN недопустим
        "классы в диапазоне": bool(sub[target_col].between(0, NUM_CLASSES - 1).all()),
        "различных классов": int(sub[target_col].nunique()),               # см. разбор выше
    }
    if test_data is not None and sample_path is not None and Path(sample_path).is_file():
        sample = pd.read_csv(sample_path)
        checks["строк как в sample"] = len(sub) == len(sample)
        checks["идентификаторы как в sample"] = bool(                       # множествами: порядок не важен
            set(sub[id_col].astype(str)) == set(sample[sample.columns[0]].astype(str))
        )
    for name, value in checks.items():
        print(f"  {name}: {value}")

    plt.figure(figsize=(12, 3))
    plt.bar(*np.unique(sub[target_col], return_counts=True), width=1.0)   # * распаковывает (значения, частоты)
    plt.title("Распределение предсказанных классов")
    plt.xlabel("класс")
    plt.tight_layout()
    plt.show()

## 20. Сводка решений

| Решение | Выбрано | Основная причина |
|---|---|---|
| разрешение | 64×64, родное | ресайз вверх не добавляет информации, но умножает время и память |
| тип массива | `uint8` | 274 МиБ против 1.07 ГиБ, а Keras всё равно приведёт батч к float |
| метка класса | `int(имя папки)` | `image_dataset_from_directory` назначает метки по строковой сортировке: папка `10` получила бы метку 2 |
| нормализация | слоем `Rescaling` в модели | применяется и при обучении, и при `predict` — рассинхронизировать невозможно |
| аугментация | слои `Random*`, без отражений | считается на GPU, сама отключается на инференсе; отражённая буква — неверная метка |
| заливка после сдвига | `constant`, ноль | фон датасета чёрный; дефолтный `reflect` затащил бы в кадр обрывки штриха |
| свёртки | две 3×3 вместо одной 5×5 | то же поле обзора, 18 496 параметров против 25 632, плюс нелинейность между ними |
| голова | `Flatten`, без узких слоёв | положение штрихов важно, а `Dense(2)` перед выходом полностью убивает обучение |
| нормализация активаций | BatchNorm, `momentum=0.9` | с дефолтным `0.99` валидация показывает `1/78` первые тысячи шагов |
| функция потерь | `sparse_categorical_crossentropy` | не нужна матрица `N×78` и не нужно хардкодить число классов |
| оптимизатор | Adam `1e-3` | работает без подбора гиперпараметров |
| разбиение | 80 / 10 / 10 со `stratify` | честная оценка на нетронутом тесте, все классы во всех частях |
| остановка | `EarlyStopping(restore_best_weights=True)` | число эпох заранее не угадать, а последние веса обычно переобучены |
| TTA | `np.roll` на ±2 пикселя | детерминированно, поэтому результат воспроизводится |
| submission | сопоставление по идентификатору | порядок чтения файлов лексикографический и не совпадает с `sample_submission` |

## Что попробовать дальше

1. **Доучиться на всех данных.** `val` и `test` — это ещё 20 % изображений.
   Число эпох взять из `best_epoch` основного прогона. В `mashtots_kaggle.ipynb`
   для этого есть флаг `REFIT_ON_ALL`.
2. **Ансамбль** из 3–5 прогонов с разными `SEED` и усреднением вероятностей.
3. **Резидуальные блоки** вместо простых `Conv-BN-ReLU`: на 70 тысячах
   изображений более глубокая сеть уже окупается.
4. **`CosineDecay`** вместо `ReduceLROnPlateau`, когда число эпох уже понятно из
   первого прогона.
5. **`label_smoothing=0.05`** через `CategoricalCrossentropy` — в рукописном
   тексте есть объективно неоднозначные образцы.
6. **Двухступенчатая схема** для пар «заглавная и строчная одной буквы», если
   confusion matrix покажет, что основные потери именно там.